(ikin)=
# Cinemática inversa

El problema de la **cinemática inversa** consiste en determinar las coordenadas articulares de un manipulador a partir de una posición u orientación deseada para su elemento terminal. A diferencia de la cinemática directa, en la que se conocen las coordenadas articulares y se calcula la pose resultante, aquí se parte de una tarea especificada en el espacio operacional para buscar una o más configuraciones que la satisfagan.

La cinemática inversa depende de la geometría del manipulador y de las restricciones impuestas a la tarea. Una misma pose puede alcanzarse con configuraciones distintas, no ser alcanzable o, en ciertas circunstancias, admitir infinitas configuraciones. Por ello, resolver el problema no consiste solamente en despejar incógnitas: también interesa determinar qué soluciones existen y cuáles son admisibles.

## Formulación del problema y tipos de solución

Para un manipulador serial de `n` articulaciones, representamos las coordenadas articulares mediante $\vec{q}=[q_1,q_2,\ldots,q_n]^T$. Si se especifica una **pose completa** deseada, el problema se expresa como

$$
T_n^0(\vec{q})=H,
$$ (ik_eq)

donde $T_n^0(\vec{q})$ es la transformación obtenida por cinemática directa y $H$ contiene la posición y orientación deseadas respecto del sistema base. Buscamos valores de $\vec{q}$ dentro del conjunto de configuraciones admisibles que satisfagan esta igualdad.

No todas las tareas exigen una pose completa. Si únicamente se prescribe la posición del extremo, basta imponer

$$
\vec{p}(\vec{q})=\vec{p}_d.
$$

En otras aplicaciones se especifican la posición y **una componente de orientación**, como ocurre con ciertos manipuladores planos o con el robot SCARA. Las ecuaciones deben corresponder a las variables operacionales que realmente se desean controlar.

Aunque la igualdad entre dos transformaciones homogéneas puede escribirse comparando sus entradas, la pose espacial contiene **seis parámetros independientes**: tres de posición y tres de orientación. Las entradas de la matriz de rotación están relacionadas por las condiciones de ortogonalidad y determinante unitario; por tanto, no constituyen nueve restricciones independientes.

Según el objetivo y la geometría del robot, pueden presentarse distintas situaciones:

- **Sin solución:** el objetivo es inalcanzable, ya sea por la geometría del mecanismo o por sus límites articulares.
- **Con una o varias soluciones discretas:** distintas configuraciones articulares pueden producir la misma posición o pose, como sucede con las configuraciones de codo arriba y codo abajo.
- **Con infinitas soluciones:** algunas tareas dejan grados de libertad sin determinar, como ocurre en ciertos manipuladores redundantes o en configuraciones geométricas especiales.

Las soluciones deben interpretarse teniendo en cuenta los **límites articulares** y, cuando corresponda, las singularidades. El hecho de encontrar una solución matemática no garantiza que sea una configuración físicamente admisible.

En estos apuntes distinguiremos dos familias de procedimientos:

1. **Métodos de solución cerrada:** emplean relaciones algebraicas, trigonométricas o construcciones geométricas para obtener expresiones de las coordenadas articulares. Estudiaremos por separado los enfoques algebraico y geométrico por su utilidad didáctica.
2. **Métodos numéricos:** buscan soluciones aproximadas mediante procedimientos iterativos. Su desarrollo se relaciona estrechamente con la cinemática diferencial, que estudiaremos posteriormente.

No existe un procedimiento de solución cerrada aplicable de manera general a cualquier manipulador. La elección del método depende de la estructura cinemática y de las características de la tarea.

## Método algebraico

En el enfoque algebraico partimos del modelo de cinemática directa y formulamos ecuaciones escalares a partir de las componentes de posición y, cuando sea necesario, de orientación. Por ejemplo, en una tarea de pose completa buscamos las coordenadas articulares que verifiquen la ecuación {eq}`ik_eq`.

Las ecuaciones resultantes suelen ser no lineales y contener funciones trigonométricas. Para resolverlas podemos emplear identidades trigonométricas, eliminación de variables y funciones trigonométricas inversas. Es importante conservar los **cuadrantes** y considerar las distintas ramas de solución; por esa razón utilizaremos `\atantwo{y}{x}` cuando debamos reconstruir un ángulo a partir de sus componentes seno y coseno.

En manipuladores de pocos grados de libertad suele ser posible obtener fórmulas compactas. Los ejemplos siguientes ilustran primero una tarea de posición en un robot RR plano y después una tarea de posición y orientación en un robot RRR plano.

:::{div}
:class: example
    
**Ejemplo.** *Calcule la cinemática inversa del manipulador RR mostrado en la Figura {numref}`ik_rr_robot_dh`. Considere que se requiere que el extremo del manipulador se posicione en las coordenadas $ (P_x, P_y) $*.

```{figure} img/ikin/rr_robot_dh.svg
:name: ik_rr_robot_dh
:align: center
:width: 50%
Manipulador RR
```

**Solución:**

Naturalmente, las longitudes de los eslabones ($ l_1 $ y $ l_2 $) del robot son valores constantes y por lo tanto se asumen conocidos. Para resolver la cinemática inversa por el método analítico debemos primero calcular la matriz que describe la cinemática directa. En este caso $ T_2^0 $ está dada por:

$$ T_2^0 = \begin{bmatrix}
c_{12} & -s_{12} & 0 & l_1 c_1 + l_2 c_{12} \\	
s_{12} & c_{12} & 0 & l_1 s_1 + l_2 s_{12} \\	
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1 
\end{bmatrix} $$

Conocida $T_2^0$, igualamos sus componentes de posición con las coordenadas deseadas $(P_x,P_y)$. Obtenemos las ecuaciones:

$$ 
l_1 c_1 + l_2 c_{12} = P_x 
$$ (ik_rr_example_eqx)

$$
l_1 s_1 + l_2 s_{12} = P_y
$$ (ik_rr_example_eqy)

Al elevar al cuadrado ambas ecuaciones y sumar, resulta

$$
P_x^2+P_y^2
=l_1^2+l_2^2
+2l_1l_2\left(c_1c_{12}+s_1s_{12}\right).
$$

Aplicando la identidad $\cos(A-B)=\cos A\cos B+\sin A\sin B$:

$$
P_x^2+P_y^2=l_1^2+l_2^2+2l_1l_2\cos q_2.
$$

Definimos

$$
K=\frac{P_x^2+P_y^2-l_1^2-l_2^2}{2l_1l_2}=\cos q_2.
$$

Como $\sin q_2=\pm\sqrt{1-K^2}$, obtenemos las **dos ramas** de solución:

$$
\boxed{q_2=\atantwo{\pm\sqrt{1-K^2}}{K}}.
$$

Para que exista una solución real es necesario que $|K|\leq 1$, suponiendo $l_1,l_2>0$. Esta condición es equivalente a que la distancia $r=\sqrt{P_x^2+P_y^2}$ satisfaga

$$
|l_1-l_2|\leq r\leq l_1+l_2.
$$

Para calcular $q_1$, retomamos las ecuaciones {eq}`ik_rr_example_eqx` y {eq}`ik_rr_example_eqy`. Al multiplicar en cruz las expresiones de posición obtenemos

$$
P_y\left(l_1 c_1+l_2 c_{12}\right)
=
P_x\left(l_1 s_1+l_2 s_{12}\right).
$$

Aplicando las identidades trigonométricas de suma de ángulos:

$$
P_y\left(l_1 c_1+l_2 c_1 c_2-l_2 s_1 s_2\right)
=
P_x\left(l_1 s_1+l_2 s_1 c_2+l_2 c_1 s_2\right).
$$

Agrupando los términos que dependen de $\cos q_1$ y $\sin q_1$:

$$
c_1\left(P_y l_1+P_y l_2 c_2-P_x l_2 s_2\right)
=
s_1\left(P_x l_1+P_x l_2 c_2+P_y l_2 s_2\right).
$$

Para recuperar el ángulo sin perder información del cuadrante, utilizamos las dos cantidades

$$
\begin{aligned}
N &= P_y(l_1+l_2 c_2)-P_x l_2 s_2,\\
D &= P_x(l_1+l_2 c_2)+P_y l_2 s_2.
\end{aligned}
$$

De las ecuaciones originales de posición se comprueba que

$$
N=r^2\sin q_1,\qquad D=r^2\cos q_1,
\qquad r^2=P_x^2+P_y^2.
$$

Por lo tanto, si $r\neq 0$, obtenemos la solución algebraica:

$$
\boxed{
q_1=\atantwo{
P_y l_1+P_y l_2 c_2-P_x l_2 s_2
}{
P_x l_1+P_x l_2 c_2+P_y l_2 s_2
}.
}
$$

**Una forma alternativa de interpretar el resultado.** Si reescribimos las ecuaciones de posición como

$$
\begin{bmatrix}P_x\\P_y\end{bmatrix}
=
\begin{bmatrix}
c_1&-s_1\\s_1&c_1
\end{bmatrix}
\begin{bmatrix}
l_1+l_2c_2\\l_2s_2
\end{bmatrix},
$$

observamos que el vector de posición resulta de rotar un ángulo $q_1$ el vector de componentes $(l_1+l_2c_2,\;l_2s_2)$. Esta interpretación conduce a la expresión equivalente

$$
q_1=\atantwo{P_y}{P_x}
-\atantwo{l_2\sin q_2}{l_1+l_2\cos q_2}.
$$

Para cada valor de $q_2$ debe utilizarse el valor correspondiente de $q_1$. En general, las dos ramas habituales coinciden geométricamente cuando $|K|=1$. Existe un caso especial: si $l_1=l_2$ y $P_x=P_y=0$, entonces $q_2=\pi$, pero $q_1$ puede tomar cualquier valor; en ese caso no debemos evaluar ninguna de las expresiones anteriores con argumentos $(0,0)$ en la función `arctan2`.

:::{note} Periodicidad de los ángulos
En una articulación revoluta, los ángulos que difieren en una vuelta completa ($2\pi$ radianes o $360^\circ$) describen la misma orientación geométrica. En los ejemplos presentaremos las soluciones angulares sin repetir esta aclaración en cada expresión; al interpretarlas se deben considerar los intervalos y límites articulares del manipulador.
:::

Podemos comprobar cada solución sustituyéndola en las ecuaciones {eq}`ik_rr_example_eqx` y {eq}`ik_rr_example_eqy`.

:::

---

:::{div}
:class: example

**Ejemplo.** Calcule la cinemática inversa del manipulador RRR de la Figura {numref}`ik_rrr_robot_dh`. El extremo debe alcanzar la posición $(P_x,P_y)$ y el eje $x_3$ debe formar un **ángulo orientado** $\phi$ respecto de $x_0$, medido alrededor del eje perpendicular al plano del manipulador.

```{figure} img/ikin/problems/rrr_robot_dh.svg
:name: ik_rrr_robot_dh
:align: center
:width: 50%
Manipulador RRR plano
```

**Solución:**

La cinemática directa del manipulador está dada por

$$
T_3^0=
\begin{bmatrix}
c_{123}&-s_{123}&0&l_1c_1+l_2c_{12}+l_3c_{123}\\
s_{123}&c_{123}&0&l_1s_1+l_2s_{12}+l_3s_{123}\\
0&0&1&0\\
0&0&0&1
\end{bmatrix}.
$$

La posición deseada proporciona dos ecuaciones:

$$
l_1c_1+l_2c_{12}+l_3c_{123}=P_x,
$$ (rrr_planar_example_ik_01)

$$
l_1s_1+l_2s_{12}+l_3s_{123}=P_y.
$$ (rrr_planar_example_ik_02)

Como tenemos tres incógnitas articulares, la información de posición no basta para obtener una solución discreta. La orientación del extremo introduce la condición

$$
\begin{bmatrix}c_{123}&-s_{123}\\s_{123}&c_{123}\end{bmatrix}
=
\begin{bmatrix}\cos\phi&-\sin\phi\\\sin\phi&\cos\phi\end{bmatrix},
$$

de la cual se deduce, **sin dividir por senos ni cosenos**, que

$$
q_1+q_2+q_3=\phi.
$$ (rrr_planar_example_ik_03)

Por tanto, $c_{123}=\cos\phi$ y $s_{123}=\sin\phi$. Definimos el punto auxiliar $Q$, correspondiente al extremo del segundo eslabón:

$$
Q_x=P_x-l_3\cos\phi,\qquad
Q_y=P_y-l_3\sin\phi.
$$

Al sustituir en las ecuaciones de posición obtenemos

$$
l_1c_1+l_2c_{12}=Q_x,
$$ (rrr_planar_example_ik_01b)

$$
l_1s_1+l_2s_{12}=Q_y.
$$ (rrr_planar_example_ik_02b)

Estas expresiones son idénticas a las de un manipulador RR plano cuyo extremo debe alcanzar $(Q_x,Q_y)$. Con

$$
K=\frac{Q_x^2+Q_y^2-l_1^2-l_2^2}{2l_1l_2},
$$

las ramas de solución, cuando $|K|\leq1$, son

$$
\boxed{q_2=\atantwo{\pm\sqrt{1-K^2}}{K}},
$$

$$
\boxed{
q_1=\atantwo{Q_y}{Q_x}
-\atantwo{l_2\sin q_2}{l_1+l_2\cos q_2},
}
$$

y finalmente

$$
\boxed{q_3=\phi-q_1-q_2}.
$$

El signo elegido para $q_2$ determina la rama y debe mantenerse al calcular $q_1$ y $q_3$. Como en el ejemplo RR, existen configuraciones degeneradas que requieren un tratamiento especial y las soluciones deben verificarse mediante la cinemática directa.

:::

## Cinemática inversa de posición: método geométrico

En el **método geométrico** aprovechamos la disposición de los ejes articulares y las dimensiones conocidas de los eslabones para relacionar las variables articulares con la posición deseada. Es frecuente utilizar triángulos, proyecciones sobre planos y relaciones trigonométricas, como la ley de cosenos.

La elección del plano de proyección **no sigue una regla universal basada únicamente en el tipo de articulación**. Buscamos el plano en el que se aprecien mejor los movimientos y las longitudes relevantes. Por ejemplo, en un manipulador plano RR todas las articulaciones revolutas giran alrededor de ejes paralelos y podemos trabajar directamente en el plano de movimiento; en ciertos manipuladores espaciales conviene utilizar varias proyecciones.

Es recomendable construir diagramas en configuraciones suficientemente generales para distinguir longitudes y ángulos, pero no es incorrecto representar una configuración con ángulos múltiplos de $90^\circ$ o desplazamientos nulos. Lo importante es no confundir una posición elegida para dibujar el mecanismo con una restricción de su movimiento.

El enfoque geométrico permite obtener expresiones compactas cuando la arquitectura del robot favorece estas construcciones. Aunque aquí lo utilizaremos principalmente para **cinemática inversa de posición**, también puede ayudar a resolver aspectos de orientación y forma parte de los métodos de solución cerrada.

:::{div}
:class: example

**Ejemplo.** Utilice el método geométrico para determinar las coordenadas articulares del manipulador RR de la Figura {numref}`ik_rr_robot_dh`, de manera que su extremo alcance el punto $(P_x,P_y)$.

**Solución.** En el ejemplo algebraico obtuvimos las ecuaciones generales del manipulador. Ahora interpretaremos geométricamente sus dos ramas de solución. Si el objetivo está dentro del espacio alcanzable y no se presenta una configuración degenerada, el robot puede adoptar las configuraciones denominadas **codo abajo** y **codo arriba**, ilustradas en la Figura {numref}`multiple_solutions`.

```{figure} img/ikin/multiple_solutions.svg
:name: multiple_solutions
:align: center
:width: 50%
Configuraciones codo arriba y codo abajo del manipulador RR.
```

Como los dos eslabones se mueven en el mismo plano, basta estudiar una proyección en el plano de movimiento. En los diagramas utilizaremos los ángulos $\alpha$ y $\beta$ para describir la dirección del punto objetivo y la geometría del triángulo formado por los eslabones.

Definimos la distancia del origen al punto objetivo y el ángulo orientado de ese vector:

$$
r=\sqrt{P_x^2+P_y^2},\qquad
\alpha=\atantwo{P_y}{P_x}.
$$

Al aplicar la ley de cosenos al triángulo de lados $l_1$, $l_2$ y $r$, obtenemos

$$
l_2^2=l_1^2+r^2-2l_1r\cos\beta,
$$

de donde

$$
\beta=\arccos\left(\frac{l_1^2+r^2-l_2^2}{2l_1r}\right).
$$

Esta construcción requiere un triángulo posible, es decir, $|l_1-l_2|\leq r\leq l_1+l_2$, y supone $r>0$ para determinar $\beta$. La ley de cosenos también proporciona

$$
r^2=l_1^2+l_2^2+2l_1l_2\cos q_2.
$$

Definamos, como en el método algebraico,

$$
K=\frac{r^2-l_1^2-l_2^2}{2l_1l_2}.
$$

**Configuración de codo abajo**

En la Figura {numref}`rr_robot_ik` se representa esta configuración y se identifican los ángulos $\alpha$ y $\beta$.

```{figure} img/ikin/rr_robot_ik.svg
:name: rr_robot_ik
:align: center
:width: 50%
Construcción geométrica para la configuración de codo abajo.
```

De la construcción geométrica se obtiene $q_1=\alpha-\beta$. El ángulo $q_2$ es positivo de acuerdo con el sentido de giro definido para la segunda articulación. Por tanto,

$$
\boxed{q_1=\alpha-\beta,\qquad q_2=\arccos K.}
$$

**Configuración de codo arriba**

La Figura {numref}`rr_robot_ik_eu` muestra la segunda configuración. El ángulo de la primera articulación corresponde ahora a la suma $\alpha+\beta$ y el giro de la segunda articulación cambia de signo:

```{figure} img/ikin/rr_robot_ik_eu.svg
:name: rr_robot_ik_eu
:align: center
:width: 50%
Construcción geométrica para la configuración de codo arriba.
```

$$
\boxed{q_1=\alpha+\beta,\qquad q_2=-\arccos K.}
$$

Así, las dos soluciones aparecen mediante un mismo triángulo y una elección consistente de signos. **Los nombres «codo arriba» y «codo abajo» se refieren a las configuraciones dibujadas**, no a una regla universal para el signo de una articulación.

En la frontera del espacio alcanzable las dos ramas pueden coincidir. Si $l_1=l_2$ y el objetivo es el origen ($r=0$), el triángulo anterior deja de determinar $\alpha$ y $\beta$: en esa situación $q_2=\pi$ y $q_1$ queda libre. Cada resultado debe comprobarse sustituyéndolo en la cinemática directa.

:::

---

:::{div}
:class: example

**Ejemplo.** *Calcule la cinemática inversa del manipulador $RRR$ mostrado en la figura. Considere que el elemento terminal deberá alcanzar la posición dada por: $ \vec{r}_P = [P_x, P_y, P_z]^T $.*

```{figure} img/ikin/examples/example_rrr_robot_dh.svg
:name: example_rrr_robot_dh
:align: center
:width: 50%
Manipulador RRR
```

**Solución:**

Para aprovechar la geometría del manipulador, primero observaremos su proyección horizontal, que permite determinar $q_1$, y después una proyección sobre el plano que contiene el movimiento de los dos eslabones restantes. Los planos $x_1y_1$ y $x_2y_2$ representados son paralelos. En la Figura {numref}`rrr_robot_iso` se puede observar una representación tridimensional del manipulador RRR; del lado izquierdo se muestra en color rojizo el plano de proyección $x_0 y_0$, y del lado derecho en color verde claro se muestra el plano de proyección $x_1 y_1$.

```{figure} img/ikin/examples/rrr_robot_iso.svg
:name: rrr_robot_iso
:align: center
:width: 80%
Vista tridimensional de los planos de proyección del manipulador RRR
```

La proyección horizontal de la Figura {numref}`rrr_robot_ik_01` permite determinar el ángulo de la base, para la rama representada:

$$ q_1 = \atantwo{P_y}{P_x} $$

Además, el valor de la distancia $a$ sería:

$$ a = \sqrt{P_x^2 + P_y^2 }  $$

```{figure} img/ikin/examples/rrr_robot_ik_01.svg
:name: rrr_robot_ik_01
:align: center
:width: 50%
Manipulador RRR, proyección en $x_0 y_0$
```

La proyección de la Figura {numref}`rrr_robot_ik_02` reduce el resto del problema a la geometría de un manipulador RR. Identificamos dos triángulos:

* Triángulo 1: el triángulo rectángulo, cuyos catetos son $a$ y $(P_z - d_1)$, y siendo $r$ la hipotenusa.
* Triángulo 2: el triángulo obtusángulo, cuyos lados son $l_2$, $l_3$ y $r$.

```{figure} img/ikin/examples/rrr_robot_ik_02.svg
:name: rrr_robot_ik_02
:align: center
:width: 50%
Manipulador RRR, proyección en $x_1 y_1$
```

Para la configuración mostrada, la segunda articulación queda descrita por la diferencia:

$$ q_2 =  \alpha - \beta $$

Utilizando el triángulo $1$ podemos calcular el valor de $\alpha$ y $r$ de la siguiente manera:

$$ \alpha = \atantwo{P_z-d_1}{a} $$
$$ r = \sqrt{a^2 + (P_z - d_1)^2} $$

En el triángulo $2$ podemos aplicar ley de cosenos para calcular a $ \beta $:

$$ l_3^2 = l_2^2 + r^2 - 2 l_2 r \cos \beta $$
$$  \beta = \arccos \left(  \frac{l_2^2 + r^2 - l_3^2}{2 l_2 r} \right) $$

Utilizando el mismo triángulo $2$ podemos ahora calcular el valor de $q_3$, aplicando también ley de cosenos:

$$ r^2 = l_2^2 + l_3^2 + 2 l_2 l_3 \cos q_3 $$
$$ q_3 = \arccos\left( \frac{r^2 - l_2^2 - l_3^2}{2 l_2 l_3} \right) $$

Las relaciones anteriores describen **la configuración representada** en los diagramas. Antes de aplicarlas, debemos comprobar que el punto objetivo sea geométricamente alcanzable. Para el triángulo de lados $l_2$, $l_3$ y $r$ se requiere

$
|l_2-l_3|\leq r\leq l_2+l_3.
$

La misma construcción puede admitir otra configuración del par de eslabones $l_2$ y $l_3$. Al cambiar de rama, debemos volver a interpretar los signos de $q_2$ y $q_3$ según los sentidos positivos establecidos por el diagrama cinemático; no basta con cambiar arbitrariamente el signo de uno de los ángulos. Si $r=0$, las fórmulas que dividen entre $r$ dejan de ser aplicables y es necesario estudiar el caso geométrico por separado.

Este ejemplo ilustra que resolver la cinemática inversa geométrica implica **seleccionar una configuración, aplicar las relaciones trigonométricas adecuadas y comprobarla con la cinemática directa**.


:::

## Desacoplo cinemático

La mayoría de manipuladores industriales de 6 grados de libertad, como el que se muestra en la Figura {numref}`abb-irb-120`, están conformados por una morfología base (RRR), a la cual se le adiciona un mecanismo de muñeca esférica (RRR). Para este tipo de manipuladores es posible aplicar una técnica conocida como desacoplo cinemático, la cual consiste en dividir el problema de la cinemática inversa en dos subproblemas: cinemática inversa de posición y cinemática inversa de orientación. La cinemática inversa de posición se suele abordar utilizando un enfoque geométrico y la cinemática inversa de orientación mediante un procedimiento analítico. 

```{figure} img/ikin/abb-irb-120.jpg
:name: abb-irb-120
:align: center
:width: 50%
Robot industrial ABB IRB 120
```

A continuación, vamos a describir una serie de pasos necesarios para realizar cinemática inversa de un manipulador de 6 grados de libertad utilizando la técnica de desacoplo cinemático. Para lo subsiguiente asumimos que se conoce de entrada la matriz de transformación homogénea $H$ que describe la posición y orientación deseada del extremo del manipulador.

Lo primero que hacemos es construir el diagrama cinemático del manipulador, basándonos en los manuales y esquemáticos del fabricante. Es importante identificar correctamente las distancias entre los ejes articulares. Con el diagrama ya realizado, procedemos a establecer los sistemas de referencia siguiendo la metodología de Denavit-Hartenberg. Obtenemos la tabla de parámetros y calculamos las matrices $ R_3^0 $ y $ R_6^3 $. Es importante comprender que $ R_3^0 $ será una matriz que depende de los primeras tres variables articulares ( $ R_3^0 (q_1,q_2,q_3) $ ), y que $ R_6^3 $ depende de las últimas tres $ R_6^3 (q_4,q_5,q_6) $.

Una vez hecho lo anterior, comenzamos a calcular las coordenadas del centro de la muñeca esférica, utilizando la siguiente ecuación:

$$ \vec{r}_m = \vec{r}_e - d_6 \vec{z}_6^0 $$

Donde $\vec{r}_e$ son las coordenadas de posición que deberá alcanzar el extremo del manipulador, $d_6$ la distancia entre los ejes $x_5$ y $x_6$, medida en la dirección del eje $ z_5 $, y $\vec{z}_6^0$ la orientación del eje $z$ del sistema de referencia del elemento terminal con respecto al de la base. Los términos $ \vec{r}_e $ y $ \vec{z}_6^0 $ se obtienen de la matriz $H$, de manera específica:

$$ \vec{r}_e = H_{1:3,4} $$
$$ \vec{z}_6^0 = H_{1:3,3} $$

Ahora procedemos a resolver la cinemática inversa de posición utilizando el método geométrico y referenciando todo el procedimiento a las coordenadas del centro de la muñeca esférica. Con esto calcularemos las tres primeras variables articulares ($q_1$, $q_2$ y $q_3$).

Con los valores de $q_1 $, $q_2$ y $q_3$ ya calculados, procedemos a determinar de forma numérica la matriz de rotación $R_3^0$, sustituyendo los valores numéricos calculados previamente.

De la cinemática directa sabemos que:

$$ R_6^0 = R_3^0 R_6^3 $$

Lo cual implica que:

$$ R_6^3 = (R_3^0)^{-1} R_6^0 $$

Donde $R_6^0$ es la submatriz de rotación de $H$, $ R_3^0 $ la matriz numérica calculada previamente y $R_6^3$  la matriz de rotación en forma simbólica calculada al inicio del procedimiento. La expresión anterior define una ecuación matricial que deberemos resolver de manera analítica, y con lo cual calcularemos los valores para las tres variables articulares restantes ($q_4$, $q_5$ y $q_6$). Esta parte es lo que corresponde a la *cinemática inversa de orientación*. Usualmente deberemos trabajar con las ecuaciones que resulten de igualar los elementos correspondientes a la tercera fila y a la tercera columna: $r_{31}$, $r_{32}$, $r_{33}$, $r_{23}$, $r_{13}$; dado que suelen quedar expresiones más sencillas de manipular.

Para comprobar los resultados podemos sustituir los valores calculados en la matriz $T_6^0$ de la cinemática directa y verificar que sea igual a la matriz $H$.


:::{div}
:class: example

**Ejemplo.** Calcula la cinemática inversa del manipulador industrial ABB IRB 140 (ver Figura {numref}`abb-irb-140-dimensions`). Considera que se requiere que el extremo del robot esté posicionado y orientado de acuerdo con la siguiente matriz $H$:

$$
H = \begin{bmatrix}
1 & 0 & 0 & 400 \\
0 & -1 & 0 & 500 \\
0 & 0 & -1 & 650 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

En la Figura {numref}`ik_abb_irb_140_kinematic_diagram` se puede observar el diagrama cinemático del manipulador y los sistemas de referencia de Denavit-Hartenberg. 

```{figure} img/fkin/abb-irb-140-kinematic-diagram.svg
:name: ik_abb_irb_140_kinematic_diagram
:align: center
:width: 60%
Diagrama cinemático del manipulador ABB IRB 140
```

En la siguiente tabla se muestran los parámetros de Denavit-Hartenberg correspondientes. Recuerda que este procedimiento de cinemática directa para el IRB 140 fue desarrollado en un ejemplo del capítulo de cinemática directa.

| $i$ | $a_i$ | $\alpha_i$ | $d_i$ | $\theta_i$ |
|:---:|:-----:|:----------:|:-----:|:----------:|
| $1$ | $70$ | $90°$ | $352$ | $q_1$ |
| $2$ | $360$ | $0°$ | $0$ | $q_2$ |
| $3$ | $0$ | $90°$ | $0$ | $q_3$ |
| $4$ | $0$ | $-90°$ | $380$ | $q_4$ |
| $5$ | $0$ | $90°$ | $0$ | $q_5$ |
| $6$ | $0$ | $0°$ | $65$ | $q_6$ |

Con esta información podemos calcular las matrices $ R_3^0 $ y $ R_6^3 $, las cuales se muestran enseguida:

$$
R_3^0 = \left[\begin{matrix}c_{1} c_{23} & s_{1} & c_{1} s_{23}\\c_{23} s_{1} & - c_{1} & s_{1} s_{23}\\s_{23} & 0 & - c_{23}\end{matrix}\right]
$$

$$ 
R_6^3 = \begin{bmatrix}c_{4} c_{5} c_{6} - s_{4} s_{6} & - c_{4} c_{5} s_{6} - c_{6} s_{4} & c_{4} s_{5}\\c_{4} s_{6} + c_{5} c_{6} s_{4} & c_{4} c_{6} - c_{5} s_{4} s_{6} & s_{4} s_{5}\\- c_{6} s_{5} & s_{5} s_{6} & c_{5}\end{bmatrix}
$$

Una vez calculadas las matrices $R_3^0$ y $R_6^3$, procedemos a determinar las coordenadas del centro de la muñeca esférica, lo cual podemos hacer mediante la siguiente expresión:

$$ 
\vec{r}_m = \vec{r}_e - d_6 \vec{z}_6^0 =
\colvec{400}{500}{650} - 65 \colvec{0}{0}{-1} = 
\colvec{400}{500}{715}
$$

Con las coordenadas del centro de la muñeca esférica ya conocidas, procedemos a realizar la cinemática inversa de posición utilizando el método geométrico, con la finalidad de calcular las primeras tres variables articulares ($q_1$, $q_2$, $q_3$). De acuerdo con lo que aprendimos en secciones previas, si queremos calcular a $q_1$ deberíamos proyectar en el plano $x_0 y_0$. En la Figura {numref}`abb_irb_140_ik_01` se puede observar la proyección del manipulador IRB 140 en el plano $x_0 y_0$. El valor de $q_1$ se puede determinar de forma sencilla de la siguiente manera:

$$ q_1 = \atantwo{500}{400} = 51.34° $$

El valor de la distancia $a$ se puede calcular mediante el teorema de Pitágoras:

$$ a = \sqrt{500^2 + 400^2} = 640.31 \text{ mm} $$

```{figure} img/ikin/abb_irb_140_ik_01.svg
:name: abb_irb_140_ik_01
:align: center
:width: 50%
Proyección en el plano $x_0 y_0$ del robot ABB IRB 140
```

Para calcular a $q_2$ debemos proyectar en $x_1 y_1$, y para $q_3$ en el plano $x_2 y_2$. Se puede verificar que estos planos son paralelos, y por lo tanto la proyección en $x_1 y_1$ nos bastaría para calcular a $q_2$ y $q_3$. En la Figura {numref}`abb_irb_140_ik_02` se puede observar la proyección del manipulador en el plano $x_1 y_1$. 

```{figure} img/ikin/abb_irb_140_ik_02.svg
:name: abb_irb_140_ik_02
:align: center
:width: 50%
Proyección en el plano $x_1 y_1$ del robot ABB IRB 140
```

Del esquema podemos notar que el valor de $q_2$ es la diferencia de los ángulos $\alpha$ y $\beta$:

$$ q_2 = \alpha - \beta $$

Ahora debemos calcular cada uno de esos valores. Del triángulo rectángulo podemos calcular la distancia (hipotenusa) $r$:

$$ r = \sqrt{ (a-70)^2 + (715-352)^2 } = 676.03 \text{ mm} $$ 

Y el ángulo $ \alpha $:

$$ \alpha = \atantwo{715-352}{a-70} = 32.48° $$

El valor del ángulo $\beta$ se puede calcular aplicando ley de cosenos en el triángulo obtusángulo:

$$ 380^2 = 360^2 + r^2 - 2(360)(r) \cos \beta $$

$$ \beta = \arccos\left( \frac{360^2 + r^2 - 380^2}{2(360)(r)} \right) = 24.7° $$

Entonces, el valor de $q_2$ sería:

$$ q_2 = 32.48° - 24.7° = 7.78° $$

Del esquema de la Figura {numref}`abb_irb_140_ik_02` se puede observar que el ángulo $q_3$ es el valor del ángulo $\phi$ más 90°, es decir:

$$ q_3 = \phi + 90° $$

Donde el ángulo $\phi$ podemos calcularlo utilizando ley de cosenos:

$$ r^2 = 380^2 + 360^2 + 2(380)(360) \cos \phi $$

$$ \phi =  \arccos\left( \frac{r^2 - 380^2 - 360^2}{2(380)(360)} \right) = 48.02° $$

Por lo tanto:

$$ q_3 = 48.02° + 90° = 138.02° $$

Una vez calculados los valores de $q_1$, $q_2$ y $q_3$, procedemos a sustituir estos valores en la matriz $ R_3^0 $, de lo cual resulta:

$$ 
R_3^0 = \begin{bmatrix}-0.5167 & 0.7809 & 0.3511\\-0.6458 & -0.6247 & 0.4389\\0.5621 & 0 & 0.8271\end{bmatrix}
$$

Ahora calculamos el producto $ \left( R_3^0 \right)^{-1} R_6^0  $:

$$ 
\left( R_3^0 \right)^{-1} R_6^0 = 
\displaystyle \left[\begin{matrix}-0.5167 & -0.6458 & 0.5621\\0.7809 & -0.6247 & 0\\0.3511 & 0.4389 & 0.8271\end{matrix}\right]
\displaystyle \left[\begin{matrix}1 & 0 & 0\\0 & -1 & 0\\0 & 0 & -1\end{matrix}\right]
=
\displaystyle \left[\begin{matrix}-0.5167 & 0.6458 & -0.5621\\0.7809 & 0.6247 & 0\\0.3511 & -0.4389 & -0.8271\end{matrix}\right]
$$

La matriz que resulta la igualamos con $R_6^3$ para formar la ecuación matricial que nos servirá para calcular la cinemática inversa de orientación, es decir:

$$ R_6^3 = \left( R_3^0 \right)^{-1} R_6^0  $$

$$ 
\displaystyle \left[\begin{matrix}c_{4} c_{5} c_{6} - s_{4} s_{6} & - c_{4} c_{5} s_{6} - c_{6} s_{4} & c_{4} s_{5}\\c_{4} s_{6} + c_{5} c_{6} s_{4} & c_{4} c_{6} - c_{5} s_{4} s_{6} & s_{4} s_{5}\\- c_{6} s_{5} & s_{5} s_{6} & c_{5}\end{matrix}\right]
=
\displaystyle \left[\begin{matrix}-0.5167 & 0.6458 & -0.5621\\0.7809 & 0.6247 & 0\\0.3511 & -0.4389 & -0.8271\end{matrix}\right]
$$

Para $ q_6 $ podemos resolver dividiendo los elementos $ r_{32} / r_{31} $ de ambas matrices e igualando, es decir:

$$
\frac{\cancel{s_5} s_6}{-c_6 \cancel{s_5}} = \frac{-0.4389}{0.3511}
$$

$$ \frac{s_6}{-c_6} = \frac{-0.4389}{0.3511} $$

$$ \frac{s_6}{c_6} = \frac{-0.4389}{-0.3511} $$

$$ q_6 = \atantwo{-0.4389}{-0.3511} = 231.34° $$

En el caso de $ q_4 $ podemos hacer $ r_{23} / r_{13} $ e igualar:

$$ \frac{s_4 \cancel{s_5}}{c_4 \cancel{s_5}} = \frac{0}{-0.5621} $$

$$ \frac{s_4}{c_4} = \frac{0}{-0.5621} $$

$$ q_4 = \atantwo{0}{-0.5621} = 180° $$

El valor de $ q_5 $ podemos determinarlo igualando los elementos $ r_{33} $ de ambas matrices:

$$ 
\cos q_5 = -0.8271 \qquad \rightarrow \qquad \sin q_5 = \sqrt{ 1 - (-0.8271)^2 }
$$

$$ 
q_5 = \atantwo{\sqrt{1 - (-0.8271)^2}}{-0.8271} = 145.8°
$$

Con esto quedan calculadas todas las posiciones articulares requeridas para que el robot ABB IRB 140 se posicione y oriente de acuerdo con la matriz $H$. Podríamos sustituir todos los valores en la matriz de la cinemática directa $T_6^0$ y obtendríamos la matriz $H$. Retomando todos los valores calculados podríamos escribir el vector de posiciones articulares $\vec{q}$ de la siguiente manera:

$$ \vec{q} = \begin{bmatrix}
51.34° \\ 7.78° \\ 138.02° \\ 180° \\ 145.8° \\ 231.34°
\end{bmatrix} $$

Es importante comprender que esta solución obtenida no es única, existen múltiples soluciones para este problema de cinemática inversa. Por ejemplo, para la cinemática inversa de orientación podemos tomar la solución para $q_5$ dada por:

$$ 
\cos q_5 = -0.8271 \qquad \rightarrow \qquad \sin q_5 = - \sqrt{ 1 - (-0.8271)^2 }
$$

$$ 
q_5 = \atantwo{-\sqrt{1 - (-0.8271)^2}}{-0.8271} = 214.2°
$$

Y entonces, esto implicaría que $q_6$ estaría dado por:

$$ \frac{-\cancel{s_5} s_6}{c_6 \cancel{s_5}} = \frac{-0.4389}{0.3511} $$

$$ \frac{-s_6}{c_6} = \frac{-0.4389}{0.3511} $$

$$ \frac{s_6}{c_6} = \frac{0.4389}{0.3511} $$

$$ q_6 = \atantwo{-0.4389}{-0.3511} = 51.34° $$

Y para el caso de $q_4$ se tendría que:

$$ \frac{-s_4 \cancel{s_5}}{-c_4 \cancel{s_5}} = \frac{0}{-0.5621} $$

$$ \frac{s_4}{c_4} = \frac{0}{0.5621} $$

$$ q_4 = \atantwo{0}{0.5621} = 0° $$

Esto nos conduciría a una segunda posible solución de la cinemática inversa dada por $\vec{q}$:

$$ \vec{q} = \begin{bmatrix}
51.34° \\ 7.78° \\ 138.02° \\ 0° \\ 214.2° \\ 51.34°
\end{bmatrix} $$

De igual manera, en la cinemática inversa de posición el manipulador se podría dibujar en una configuración diferente a la planteada, y obtendríamos un conjunto solución diferente. Se recomienda al lector que se tome el tiempo de verificar que las soluciones calculadas son correctas, haciendo la sustitución descrita en párrafos anteriores. El trabajo de multiplicación matricial para obtener $ T_6^0 $, aún cuando se haga de forma completamente numérica, puede llegar a resultar cuando menos tedioso, por lo cual se recomienda que estas operaciones se realicen con el auxilio de herramientas computacionales.


:::

(ik_num)=
## Resolviendo la cinemática inversa de forma numérica

### Método de Newton

### Descenso del gradiente

# Problemas 

---
Calcule la cinemática inversa del manipulador RRP mostrado en la Figura {numref}`rrp_robot_dh`, considere que $d_1$ es una longitud conocida. El elemento terminal deberá alcanzar la posición dada por:

$$ \vec{r}_P = \colvec{P_x}{P_y}{P_z} $$

```{figure} img/ikin/problems/rrp_robot_dh.svg
:name: rrp_robot_dh
:align: center
:width: 50%
Manipulador RRP
```

---
Calcule la cinemática inversa del manipulador RRP mostrado en la Figura {numref}`rrp_robot_dh`, considere que $d_1 = 300 $mm. El elemento terminal deberá alcanzar la posición dada por:

$$ \vec{r}_P = \colvec{250}{175}{380} $$

---
Calcule la cinemática inversa del manipulador RRP mostrado en la Figura {numref}`rrp_robot_dh`, considere que $d_1 = 280 $mm. El elemento terminal deberá alcanzar la posición dada por:

$$ \vec{r}_P = \colvec{220}{200}{350} $$

---
Calcula la cinemática inversa del manipulador RRR mostrado en la Figura {numref}`ik_rrr_robot_dh`, considerando que el elemento terminal deberá alcanzar la posición dada por:

$$ \vec{r}_P = \colvec{350}{280}{0} $$

Y orientado de tal manera que el eje $x_3$ se encuentre alineado con $x_0$. Se sabe que $l_1=l_2=200 \text{ mm}$ y $ l_3 = 150 \text{ mm} $.

---
En la Figura {numref}`ik_robot_2d_prr` se muestra un manipulador PRR que deberá alcanzar dos objetos A y B, tal que sus posiciones y orientaciones están definidas mediante los sistemas de referencia $\{A\}$ y $\{B\}$ adheridos a estos. Se sabe que:

$$ H_B^0 = 
\begin{bmatrix}
\frac{\sqrt{3}}{2} & - \frac{1}{2} & 0 & 400\\0 & 0 & -1 & 0\\\frac{1}{2} & \frac{\sqrt{3}}{2} & 0 & 300\\0 & 0 & 0 & 1
\end{bmatrix}
$$

Calcule los valores articulares requeridos para alcanzar cada uno de los objetos. Considere que $l_2 = 250$ mm y $l_3 = 200$ mm.

```{figure} img/ikin/problems/ik_robot_2d_prr.svg
:name: ik_robot_2d_prr
:align: center
:width: 50%
Manipulador PRR
```


---
En la figura se muestra el esquema de un manipulador planar PPR que deberá realizar operaciones de soldadura a lo largo de la trayectoria compuesta, esquematizada mediante la línea discontinua. Los puntos A y B denotan el inicio y fin de la trayectoria, y el elemento terminal del manipulador deberá alcanzar estos puntos con la orientación dada por los sistemas $\{A\}$ y $\{B\}$. La longitud $ l_3 $ es de 100 mm. Se sabe que:

$$ 
T_A^0 = \left[\begin{matrix}\frac{\sqrt{2}}{2} & - \frac{\sqrt{2}}{2} & 0 & 300\\0 & 0 & -1 & 0\\\frac{\sqrt{2}}{2} & \frac{\sqrt{2}}{2} & 0 & 150\\0 & 0 & 0 & 1\end{matrix}\right]
\qquad ; \qquad
T_B^0 = \left[\begin{matrix}1 & 0 & 0 & 500\\0 & 0 & -1 & 0\\0 & 1 & 0 & 250\\0 & 0 & 0 & 1\end{matrix}\right]
$$

Calcule los valores articulares requeridos para alcanzar los puntos $A$ y $B$.

```{figure} img/ikin/problems/ik_robot_2d_ppr.svg
:name: ik_robot_2d_ppr
:align: center
:width: 60%
.
```

---
Calcula la cinemática inversa del manipulador RRP mostrado en la Figura {numref}`rrp_robot_dh_with_off_ik`, considerando que el elemento terminal deberá alcanzar el punto $(200, 250, 350)$.

```{figure} img/ikin/problems/rrp_robot_dh_with_off_ik.svg
:name: rrp_robot_dh_with_off_ik
:align: center
:width: 60%
.
```

---
Para el manipulador industrial mostrado en la Figura {numref}`ik_6dof_robot`, calcule la cinemática inversa de posición y orientación si el extremo del manipulador debe estar orientado y posicionado de acuerdo con:

$$ H = \begin{bmatrix} 1 & 0 & 0 & 250\\ 0 & -1 & 0 & 300\\ 0 & 0 & -1 & 200\\0 & 0 & 0 & 1\end{bmatrix} $$

```{figure} img/ikin/problems/ik_6dof_robot.svg
:name: ik_6dof_robot
:align: center
:width: 60%
.
```

---
Para el manipulador industrial mostrado en la Figura {numref}`ik_6dof_robot`, calcule la cinemática inversa de posición y orientación si el extremo del manipulador debe estar orientado y posicionado de acuerdo con:

$$ H = \begin{bmatrix} 1 & 0 & 0 & 250\\ 0 & -1 & 0 & 300\\ 0 & 0 & -1 & 200\\0 & 0 & 0 & 1\end{bmatrix} $$

```{figure} img/ikin/problems/ik_6dof_robot_02.svg
:name: ik_6dof_robot_02
:align: center
:width: 60%
.
```

---
Utiliza la técnica del desacoplo cinemático para calcular la cinemática inversa del robot FANUC LR Mate 200iC/5L. 

## Problemas para resolver utilizando la computadora

---
Desarrolla una función denominada `rr_planar_robot_ik` que deberá calcular las dos soluciones (codo arriba y codo abajo) de la cinemática inversa del manipulador planar RR mostrado en la figura. La función deberá recibir como argumentos de entrada las coordenadas ($P_x$, $P_y$) del punto que deberá alcanzar el manipulador, las longitudes de los eslabones: $ l_1 $ y $ l_2 $, y un argumento denominado `sol` en el cual deberá indicarse la solución a calcular, se usará `'up'` para la solución de codo arriba y `'down'` para la solución de codo abajo. Enseguida se muestra una sugerencia de plantilla para la definición de la función, designación de los parámetros y de los valores a devolver.

```python
def rr_planar_robot_ik(px, py, l1, l2, sol):
	# hacer los cálculos aquí
	return q1, q2
```

---

Desarrolla una función denominada `rrr_planar_robot_ik` que calcule la cinemática inversa de un manipulador planar RRR mostrado en la Figura. La función debe recibir como argumentos de entrada las coordenadas $\left(P_x, P_y\right)$ del punto que deberá alcanzar el manipulador, el ángulo $\phi$ que describe la orientación, las longitudes de los eslabones: $l_1$, $l_2$ y $l_3$, y un argumento denominado `sol` en el cual deberá indicarse la solución a calcular, se usará `'up'` para la solución de codo arriba y `'down'` para la solución de codo abajo.

```python
def rrr_planar_robot_ik(px, py, phi, l1, l2, l3, sol):
	# hacer los cálculos aquí
	return q1, q2, q3
```